# Лабораторная работа №6: Цепочки и контекст

[Исходное задание](https://colab.research.google.com/drive/1GpBtqhLWp_1gUgEfEOXnig1vFdk1TpfU)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

#Задание семинара 6
1. Создайте 3 шаблона промптов для выполнения 3-х задач LLM GigaChat, из которых  можно составить цепочку.
2. Создайте 3 цепочки из шаблонов предыдущего пункта, из которых создайте последовательную цепочку (простейшую). Последовательную цепочку сделайте 2 способами: без использования функции SequentialChain (или подобной), использую метод invoke, и с использованием этой функции.
3. Создайте цепочку, состоящую из минимум 3-х звеньев, последняя создает промпт для генерации изображений. Затем изображение генерируется.
4. Возьмите любую программу диалога с LLM с историей. Реализуйте контекстное окно для истории диалога.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. Три шаблона и три цепочки

In [ ]:
idea_prompt = PromptTemplate.from_template("Предложи одну идею музыкального помощника по теме {topic}. До 60 слов.")
plan_prompt = PromptTemplate.from_template("Составь план реализации в 3 пунктах для идеи: {idea}. До 80 слов.")
detail_prompt = PromptTemplate.from_template("Опиши минимальный прототип по плану: {plan}. До 100 слов.")
idea_chain = idea_prompt | llm | StrOutputParser()
plan_chain = plan_prompt | llm | StrOutputParser()
detail_chain = detail_prompt | llm | StrOutputParser()
print("Созданы 3 шаблона и 3 цепочки")


## 2. Последовательность вручную и SequentialChain

In [ ]:
topic = "проверка ключевых слов в песнях"
idea = idea_chain.invoke({"topic": topic})
plan = plan_chain.invoke({"idea": idea})
detail = detail_chain.invoke({"plan": plan})
print("РУЧНАЯ ЦЕПОЧКА\nИдея:", idea, "\nПлан:", plan, "\nПрототип:", detail)

from langchain_classic.chains import LLMChain, SequentialChain
sequence = SequentialChain(chains=[
    LLMChain(llm=llm, prompt=idea_prompt, output_key="idea"),
    LLMChain(llm=llm, prompt=plan_prompt, output_key="plan"),
    LLMChain(llm=llm, prompt=detail_prompt, output_key="detail"),
], input_variables=["topic"], output_variables=["idea", "plan", "detail"])
result = sequence.invoke({"topic": topic})
for key in ["idea", "plan", "detail"]:
    print(f"\nSEQUENTIAL {key}: {result[key]}")


## 3. Три звена → промпт → изображение

In [ ]:
from lab_runtime import generate_image
concept_chain = PromptTemplate.from_template("Предложи концепцию обложки по теме {topic}, 40 слов.") | llm | StrOutputParser()
scene_chain = PromptTemplate.from_template("Опиши визуальную сцену для концепции {concept}, 50 слов.") | llm | StrOutputParser()
image_prompt_chain = PromptTemplate.from_template("Переведи сцену в английский промпт изображения, без пояснений и без текста на картинке: {scene}") | llm | StrOutputParser()
concept = concept_chain.invoke({"topic": "музыкальная студия будущего"})
scene = scene_chain.invoke({"concept": concept})
image_prompt = image_prompt_chain.invoke({"scene": scene})
print("Концепция:", concept, "\nСцена:", scene, "\nПромпт:", image_prompt)
path = generate_image(image_prompt, "lab06_chain.png")
display(Image(filename=str(path), width=600))


## 4. Диалог с контекстным окном

In [ ]:
MAX_MESSAGES = 4
history = []
system = SystemMessage(content="Ты помощник по Python. Ответ до 35 слов.")
for request in ["Меня зовут Алексей.", "Изучаю Python.", "Что такое список?", "Как добавить элемент?", "Как меня зовут?"]:
    history.append(HumanMessage(content=request))
    history = history[-MAX_MESSAGES:]
    response = llm.invoke([system] + history)
    history.append(response)
    history = history[-MAX_MESSAGES:]
    print("User:", request, "\nAssistant:", response.content, "\nВ окне:", len(history))
assert len(history) <= MAX_MESSAGES


## Вывод

Ручной invoke и SequentialChain передают результаты между тремя этапами. Ещё одна цепочка сформировала промпт и изображение. Контекстное окно ограничивает объём истории.